In [0]:
dbutils.widgets.text("catalog", "banking")
dbutils.widgets.text("schema_landing", "landing")
dbutils.widgets.text("schema_bronze", "bronze")
dbutils.widgets.text("schema_silver", "silver")
dbutils.widgets.text("schema_gold", "gold")

catalog = dbutils.widgets.get("catalog")
schema_landing = dbutils.widgets.get("schema_landing")
schema_bronze = dbutils.widgets.get("schema_bronze")
schema_silver = dbutils.widgets.get("schema_silver")
schema_gold = dbutils.widgets.get("schema_gold")

silver_credit_table = f"{catalog}.{schema_silver}.silver_credit"

gold_table_full = f"{catalog}.{schema_gold}.gold_risk_customer_summary"

In [0]:
from pyspark.sql import functions as F

silver_credit_table_df = spark.table(silver_credit_table)

result = (
    silver_credit_table_df.groupBy("risk_grade")
      .agg(
          F.count("customer_id").alias("total_customers"),
          F.avg("credit_score").alias("avg_credit_score"),
          F.sum("external_active_loans").alias("total_external_loans"),
          F.sum("external_overdue_amount").alias("total_overdue_amount")
      )
)

result.write.mode("overwrite").saveAsTable(gold_table_full)

In [0]:
count = spark.sql("""
SELECT COUNT(*) AS cnt
FROM banking.gold.gold_risk_customer_summary
""").collect()[0]["cnt"]

dbutils.notebook.exit(str(count))